## Setup the data

In [1]:
import os
import time
import numpy as np
import torch
import onnx
import onnxruntime as ort
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Export decoder to ONNX

In [ ]:
!python MobileSAM/scripts/export_onnx_model.py \
  --checkpoint immich-sticker-gen/serving/models/distilled_mobile_sam_full.pth \
  --model-type vit_t \
  --output immich-sticker-gen/serving/models/distilled_mobile_sam_decoder.onnx \
  --return-single-mask

## Convert encoder

In [ ]:
import torch
import onnx
from mobile_sam import sam_model_registry

device = torch.device("cpu")

base_ckpt = "immich-sticker-gen/serving/models/mobile_sam.pt"
model_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_full.pth"

model = sam_model_registry["vit_t"](checkpoint=base_ckpt)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

encoder = model.image_encoder
encoder.eval()

encoder_onnx_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder.onnx"
dummy_input = torch.randn(1, 3, 1024, 1024)

torch.onnx.export(
    encoder,
    dummy_input,
    encoder_onnx_path,
    export_params=True,
    opset_version=20,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["image_embeddings"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "image_embeddings": {0: "batch_size"},
    },
)

onnx_model = onnx.load(encoder_onnx_path)
onnx.checker.check_model(onnx_model)

print(f"Encoder ONNX saved to {encoder_onnx_path}")

## Load test data from SA-1B

In [2]:
SHARD_URL = "https://scontent.xx.fbcdn.net/m1/v/t6/An_YmP5OIPXun-vu3hkckAZZ2s4lPYoVkiyvCcWiVY21mu1Ng5_1HeCa2CWiSTsskj8HQ8bN013HxNpYDdSC_7jWQq_svcg.tar?_nc_gid&ccb=10-5&oh=00_Af08ZycNdYdNMlMCW5sVXhQ0W2iYlA4GO1vtsjm6IY-yYw&oe=69F57028&_nc_sid=0fdd51"

import os
import json
import requests
import tarfile
from pathlib import Path
from PIL import Image

def prepare_sa1b_subset(
    shard_url,
    out_dir="benchmark_sa1b",
    max_images=50,
    tar_path="sa1b_shard.tar",
):
    out_dir = Path(out_dir)
    images_dir = out_dir / "images"
    anns_dir = out_dir / "annotations"
    images_dir.mkdir(parents=True, exist_ok=True)
    anns_dir.mkdir(parents=True, exist_ok=True)

    # download once
    if not os.path.exists(tar_path):
        print("Downloading shard...")
        with requests.get(shard_url, stream=True) as r:
            r.raise_for_status()
            with open(tar_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("Download complete.")
    else:
        print("Shard already exists, skipping download.")

    manifest_path = out_dir / "manifest.json"
    if manifest_path.exists():
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
        if len(manifest) >= max_images:
            print(f"Subset already exists ({len(manifest)} pairs), skipping extraction.")
            return manifest[:max_images]

    print("Extracting image/json pairs...")
    saved = []
    pending_json = {}

    with tarfile.open(tar_path) as tar:
        members = tar.getmembers()

        # first pass: read jsons into memory by stem
        for member in members:
            if member.isfile() and member.name.endswith(".json"):
                stem = Path(member.name).stem
                f = tar.extractfile(member)
                if f is not None:
                    pending_json[stem] = f.read()

        count = 0

        # second pass: extract matching jpg + json
        for member in members:
            if not (member.isfile() and member.name.endswith(".jpg")):
                continue

            stem = Path(member.name).stem
            if stem not in pending_json:
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            img = Image.open(f).convert("RGB")

            img_path = images_dir / f"{stem}.jpg"
            ann_path = anns_dir / f"{stem}.json"

            img.save(img_path, quality=95)

            with open(ann_path, "wb") as jf:
                jf.write(pending_json[stem])

            saved.append({
                "id": stem,
                "image_path": str(img_path),
                "annotation_path": str(ann_path),
            })

            count += 1
            if count >= max_images:
                break

    with open(manifest_path, "w") as f:
        json.dump(saved, f, indent=2)

    print(f"Saved {len(saved)} image/json pairs to {out_dir}")
    return saved

pairs = prepare_sa1b_subset(
    shard_url=SHARD_URL,
    out_dir="benchmark_sa1b",
    max_images=50,
    tar_path="sa1b_shard.tar",
)

Shard already exists, skipping download.
Subset already exists (50 pairs), skipping extraction.


## Calculate metrics

In [ ]:
import os
import json
import time
import copy
import cv2
import numpy as np
import onnxruntime as ort
from PIL import Image
from pycocotools import mask as mask_utils

# Optional quantization imports
from onnxruntime.quantization import quantize_dynamic, quantize_static, CalibrationDataReader, QuantType

# --------------------------------------------------
# paths
# --------------------------------------------------
BASE_ENCODER = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder.onnx"
BASE_DECODER = "immich-sticker-gen/serving/models/distilled_mobile_sam_decoder.onnx"

OPT_ENCODER = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder_optimized.onnx"
DYN_ENCODER = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder_dynamic.onnx"
STATIC_ENCODER = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder_static.onnx"

# assumes you already have:
# pairs = prepare_sa1b_subset(...)

# --------------------------------------------------
# config
# --------------------------------------------------
NUM_EVAL_SAMPLES = min(50, len(pairs))
NUM_TRIALS = 100
NUM_ENCODER_THROUGHPUT_TRIALS = 100

# --------------------------------------------------
# helpers
# --------------------------------------------------
PIXEL_MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
PIXEL_STD = np.array([58.395, 57.12, 57.375], dtype=np.float32)

def load_pair(pair):
    image = np.array(Image.open(pair["image_path"]).convert("RGB"))
    with open(pair["annotation_path"], "r") as f:
        ann_data = json.load(f)
    return image, ann_data

def preprocess_sam(image_rgb, image_size=1024):
    h, w = image_rgb.shape[:2]
    scale = image_size / max(h, w)
    new_h = int(h * scale + 0.5)
    new_w = int(w * scale + 0.5)

    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    padded = np.zeros((image_size, image_size, 3), dtype=np.float32)
    padded[:new_h, :new_w, :] = resized.astype(np.float32)

    padded = (padded - PIXEL_MEAN) / PIXEL_STD
    x = np.transpose(padded, (2, 0, 1)).astype(np.float32)
    return x, (h, w)

def ann_to_mask(ann):
    seg = ann["segmentation"]
    if isinstance(seg["counts"], list):
        rle = mask_utils.frPyObjects(seg, seg["size"][0], seg["size"][1])
    else:
        rle = seg
    mask = mask_utils.decode(rle)
    if mask.ndim == 3:
        mask = mask[..., 0]
    return mask.astype(bool)

def mask_to_box(mask):
    ys, xs = np.where(mask)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)

def compute_iou(pred_mask, gt_mask):
    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return inter / union if union > 0 else 0.0

def resize_box_to_1024(box_xyxy, original_size, target_length=1024):
    h, w = original_size
    scale = target_length / max(h, w)
    return box_xyxy * scale

# --------------------------------------------------
# optimization / quantization
# --------------------------------------------------
def save_optimized_encoder(input_path, output_path):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
    so.optimized_model_filepath = output_path
    _ = ort.InferenceSession(input_path, sess_options=so, providers=["CPUExecutionProvider"])

def save_dynamic_quantized_encoder(input_path, output_path):
    quantize_dynamic(
        model_input=input_path,
        model_output=output_path,
        weight_type=QuantType.QInt8,
    )

class SAMCalibrationDataReader(CalibrationDataReader):
    def __init__(self, pairs_subset):
        self.data = []
        for pair in pairs_subset:
            image, _ = load_pair(pair)
            x, _ = preprocess_sam(image)
            x = np.ascontiguousarray(x[None, ...], dtype=np.float32)
            self.data.append({"input": x})
        self.iterator = iter(self.data)

    def get_next(self):
        return next(self.iterator, None)

def save_static_quantized_encoder(input_path, output_path, calib_pairs):
    reader = SAMCalibrationDataReader(calib_pairs)
    quantize_static(
        model_input=input_path,
        model_output=output_path,
        calibration_data_reader=reader,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
    )

# --------------------------------------------------
# create optimized models
# --------------------------------------------------
if not os.path.exists(OPT_ENCODER):
    save_optimized_encoder(BASE_ENCODER, OPT_ENCODER)

if not os.path.exists(DYN_ENCODER):
    save_dynamic_quantized_encoder(BASE_ENCODER, DYN_ENCODER)

if not os.path.exists(STATIC_ENCODER):
    calib_pairs = pairs[:min(16, len(pairs))]
    save_static_quantized_encoder(BASE_ENCODER, STATIC_ENCODER, calib_pairs)

# --------------------------------------------------
# benchmark
# --------------------------------------------------
def benchmark_model(encoder_path, decoder_path, pairs_subset, label):
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

    enc_sess = ort.InferenceSession(
        encoder_path,
        sess_options=sess_opts,
        providers=["CPUExecutionProvider"],
    )
    dec_sess = ort.InferenceSession(
        decoder_path,
        sess_options=sess_opts,
        providers=["CPUExecutionProvider"],
    )

    enc_input_name = enc_sess.get_inputs()[0].name

    def run_onnx_box_prompt(image, box_xyxy):
        x, orig_size = preprocess_sam(image)
        x = np.ascontiguousarray(x[None, ...], dtype=np.float32)

        image_embeddings = enc_sess.run(None, {enc_input_name: x})[0]

        scaled_box = resize_box_to_1024(box_xyxy.copy(), orig_size)

        point_coords = np.array(
            [[[scaled_box[0], scaled_box[1]], [scaled_box[2], scaled_box[3]]]],
            dtype=np.float32,
        )
        point_labels = np.array([[2, 3]], dtype=np.float32)

        mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32)
        has_mask_input = np.zeros((1,), dtype=np.float32)
        orig_im_size = np.array([orig_size[0], orig_size[1]], dtype=np.float32)

        ort_inputs = {
            "image_embeddings": image_embeddings,
            "point_coords": point_coords,
            "point_labels": point_labels,
            "mask_input": mask_input,
            "has_mask_input": has_mask_input,
            "orig_im_size": orig_im_size,
        }

        masks, scores, _ = dec_sess.run(None, ort_inputs)
        pred_mask = masks[0, 0] > 0.0
        return pred_mask, scores

    print(f"\n=== {label} ===")
    print(f"Execution provider: {enc_sess.get_providers()}")

    # accuracy = mean IoU
    ious = []
    for pair in pairs_subset:
        image, ann_data = load_pair(pair)
        anns = ann_data.get("annotations", [])
        if not anns:
            continue

        gt_mask = ann_to_mask(anns[0])
        box = mask_to_box(gt_mask)

        pred_mask, _ = run_onnx_box_prompt(image, box)
        ious.append(compute_iou(pred_mask, gt_mask))

    mean_iou = 100.0 * np.mean(ious) if ious else 0.0
    print(f"Mean IoU: {mean_iou:.2f}% ({len(ious)} samples)")

    # model size
    enc_size = os.path.getsize(encoder_path) / 1e6
    dec_size = os.path.getsize(decoder_path) / 1e6
    print(f"Encoder Size on Disk: {enc_size:.2f} MB")
    print(f"Decoder Size on Disk: {dec_size:.2f} MB")
    print(f"Total Size on Disk: {enc_size + dec_size:.2f} MB")

    # single sample latency
    single_image, single_ann_data = load_pair(pairs_subset[0])
    single_gt_mask = ann_to_mask(single_ann_data["annotations"][0])
    single_box = mask_to_box(single_gt_mask)

    _ = run_onnx_box_prompt(single_image, single_box)

    latencies = []
    for _ in range(NUM_TRIALS):
        t0 = time.time()
        _ = run_onnx_box_prompt(single_image, single_box)
        latencies.append(time.time() - t0)

    print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
    print(f"Inference Throughput (single sample): {NUM_TRIALS / np.sum(latencies):.2f} FPS")

    # encoder throughput, batch=1 repeated
    encoder_times = []
    images_processed = 0

    for _ in range(NUM_ENCODER_THROUGHPUT_TRIALS):
        for pair in pairs_subset:
            image, _ = load_pair(pair)
            x, _ = preprocess_sam(image)
            x = np.ascontiguousarray(x[None, ...], dtype=np.float32)

            t0 = time.time()
            _ = enc_sess.run(None, {enc_input_name: x})
            encoder_times.append(time.time() - t0)
            images_processed += 1

    encoder_fps = images_processed / np.sum(encoder_times)
    print(f"Encoder Throughput (batch=1 repeated): {encoder_fps:.2f} FPS")

# --------------------------------------------------
# run all experiments
# --------------------------------------------------
eval_pairs = pairs[:NUM_EVAL_SAMPLES]

benchmark_model(BASE_ENCODER, BASE_DECODER, eval_pairs, "Baseline ONNX")
benchmark_model(OPT_ENCODER, BASE_DECODER, eval_pairs, "Graph-Optimized ONNX")
benchmark_model(DYN_ENCODER, BASE_DECODER, eval_pairs, "Dynamic Quantized ONNX")
benchmark_model(STATIC_ENCODER, BASE_DECODER, eval_pairs, "Static Quantized ONNX")

  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
   


=== Baseline ONNX ===
Execution provider: ['CPUExecutionProvider']
Mean IoU: 80.60% (50 samples)
Encoder Size on Disk: 28.06 MB
Decoder Size on Disk: 16.51 MB
Total Size on Disk: 44.57 MB
Inference Latency (single sample, median): 302.10 ms
Inference Latency (single sample, 95th percentile): 340.78 ms
Inference Latency (single sample, 99th percentile): 354.40 ms
Inference Throughput (single sample): 3.25 FPS
